[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO_2026/blob/main/Perotti_4_EEG.ipynb)

In [ ]:
# File -> Save a copy in Drive
# Google Drive is also used as a cache for the Sleep-EDF files, so they are downloaded only once
from google.colab import auth; auth.authenticate_user()
from google.colab import drive; drive.mount('/content/drive')

# Lesson 4 - Sleep staging: the standard pipeline on one subject

**Computational Neuroscience 2026/27 · Alan Perotti**

This is our first lesson on sleep data. A night of sleep is recorded with a **polysomnography**: EEG, eye movements, muscle tone, and more, for about eight hours. An expert then reads the recording in windows of 30 seconds and gives each window a **sleep stage**. The result is a **hypnogram**: the sequence of stages across the night.

Scoring a night by hand takes hours. Today we build a model that does it automatically. We work on one person: we train on their first night and test on their second night. We go step by step, from the raw signals to a model that works and that we understand.

The split is the right one from the start: the lesson on data leakage told us why consecutive windows of the same night cannot be split at random.

## Learning goals

- Know what a polysomnography is and what the sleep stages are
- Load EDF recordings and hypnograms with MNE, and cut them into 30-second epochs
- Look at the signals, the hypnogram, and the spectrum of each stage
- Extract spectral features (relative power in each frequency band) and build a table epochs x features
- Train a random forest on one night and evaluate it on another night of the same subject
- Read the metrics when the classes are unbalanced

---

## The big picture

**The goal.** A function that takes 30 seconds of EEG and answers with a sleep stage: W, N1, N2, N3 or REM.

**The problem.** A model cannot read an EDF file. It needs a **table**: one row per example, a few numbers per row (the features), and the answer (the label). Most of the lesson turns a raw recording into such a table. Only then do we train and test, as in Lesson 3.

**The main path.** Eight stages. Each one produces what the next one needs:

| Stage | Step | From | To |
|---|---|---|---|
| 1. download | 2 | PhysioNet | two files per night on Drive: the signals and the hypnogram |
| 2. read | 3 | the files | the signals of night 1 in memory (`raw_1`), with the stages attached |
| 3. cut the wake | 4 | 22 hours of recording | only the night, plus 30 minutes on each side |
| 4. epochs | 5 | a continuous signal | about a thousand windows of 30 s, each with its stage (`epochs_1`, `epochs_2`) |
| 5. features | 8 | windows of 3000 samples | a table with 10 numbers per window (`df_1`, `df_2`) |
| 6. split | 9 | two tables | train = night 1, test = night 2 (`X_train`, `y_train`, `X_test`, `y_test`) |
| 7. train | 10 | the train table | a fitted random forest (`model`) |
| 8. test | 11 | the test table | the scores on night 2 |

**The side looks.** Steps 1, 6, 7, 12 and 13, and some plots inside the other steps, are not needed by the model. They are there to understand: what the stages are, what the data look like, why the model makes its errors. Each step starts with a box that says whether it is on the main path or a side look.

At the end, Step 14 has questions to reason about.

# Step 1 - Sleep, its stages, and the dataset

**Side look.** The model does not need this. This step is background: no code. We need it to judge whether the errors of the model make sense.

> **What we do:** introduce the sleep stages and how they look in the EEG.
>
> **Why:** the model will only see numbers. To judge whether its errors make sense, we need to know what each stage looks like.

## The sleep stages

Experts score sleep in windows (**epochs**) of 30 seconds. The modern standard (AASM) has five stages:

| Stage | What happens | What we see in the EEG |
|---|---|---|
| **W** (wake) | awake, eyes open or closed | beta activity (fast, low amplitude); alpha (8-12 Hz) with eyes closed |
| **N1** | transition from wake to sleep, a few minutes | theta (4-8 Hz), alpha disappears |
| **N2** | light sleep, about half of the night | **sleep spindles** (bursts at 11-16 Hz) and **K-complexes** (large single waves) |
| **N3** | deep sleep, slow-wave sleep | **slow waves** (delta, below 4 Hz) with high amplitude |
| **REM** | dreaming sleep | mixed activity similar to wake, rapid eye movements, no muscle tone |

A night goes through 4-6 cycles of about 90 minutes: N1 -> N2 -> N3 -> N2 -> REM. N3 is more frequent at the start of the night, REM towards the end.

## The hypnogram

The **hypnogram** is the plot of the stage against time. By convention W is at the top and N3 at the bottom. We will draw one in Step 4.

## The dataset: Sleep-EDF Expanded

We use the **Sleep Cassette** part of the [Sleep-EDF Database Expanded](https://physionet.org/content/sleep-edfx/) (PhysioNet, free access, no registration).

- Healthy subjects aged 25 to 101, recorded in 1987-1991 for a study on the effect of age on sleep. Usually two nights per subject.
- Channels: EEG Fpz-Cz, EEG Pz-Oz, horizontal EOG (eye movements), submental EMG (chin muscle tone), an event marker, and in some recordings respiration.
- The hypnograms were scored by hand with the older Rechtschaffen & Kales rules (1968): stages W, 1, 2, 3, 4, R, plus M (movement) and ? (not scored). We merge stages 3 and 4 into N3, as in the AASM standard, and we discard M and ?.

Our analysis roughly follows the MNE tutorial [Sleep stage classification from polysomnography (PSG) data](https://mne.tools/stable/auto_tutorials/clinical/60_sleep.html).

# Step 2 - The tools and the download

**Where we are.** Main path, stage 1 of 8: **[download]** -> read -> cut the wake -> epochs -> features -> split -> train -> test

We start from nothing: no library for EEG, no data. At the end of this step we have the two nights of one subject on Drive.

> **What we do:** install MNE, import the libraries, choose the subject, download the two nights.
>
> **Why:** MNE is the standard Python library for EEG. It reads the EDF files and cuts the signal into epochs.

MNE is not installed on Colab by default. The other libraries (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`) are already there.

The installation may print a warning about dependency conflicts (`ipython`, `moviepy`, `decorator`): it concerns packages we do not use, and can be ignored.

**If Colab asks to restart the session after the installation, restart it, then continue from the imports cell below.** Do not run the installation again: MNE stays installed, and Drive stays mounted.

In [ ]:
!pip install mne

## Imports

Everything we use, at the top. You do not need to understand each line now: every tool is explained when we use it.

- `os`: file paths
- `numpy`, `pandas`: numbers and tables
- `matplotlib`, `seaborn`: plots
- `mne`: reads EEG files, cuts epochs, computes spectra
- `fetch_data`: downloads the Sleep-EDF recordings
- `make_pipeline`, `RandomForestClassifier`: the model, as in Lesson 3
- the `metrics`: scores and confusion matrix

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from mne.datasets.sleep_physionet.age import fetch_data

from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')     # MNE prints a lot of messages: keep only the warnings

## Settings

All the choices we may want to change are here, at the top.

- `SUBJECT_A`: the subject we work on. Their night 1 is the train set, their night 2 the test set. A few subjects of the Sleep Cassette have only one night (subject 13 has no night 2, subjects 36 and 52 have no night 1), so we choose a subject with both nights.
- `CLASS_SCHEME`: how many classes we predict.
  - `"5"`: W, N1, N2, N3, REM (the default)
  - `"4"`: W, Light (N1 + N2), Deep (N3), REM
  - `"3"`: W, NREM (N1 + N2 + N3), REM
- `DATA_DIR`: the folder on Google Drive where the files are kept.

In [ ]:
SUBJECT_A = 0            # train on night 1, test on night 2 of this subject
CLASS_SCHEME = "5"       # "5", "4" or "3"
DATA_DIR = '/content/drive/MyDrive/sleep_edf_cache'   # the downloaded files are kept here

# Create the folder on Drive, if it does not exist yet
os.makedirs(DATA_DIR, exist_ok=True)

# Subjects with only one night in the Sleep Cassette: avoid them as SUBJECT_A
ONE_NIGHT_ONLY = [13, 36, 52]
assert SUBJECT_A not in ONE_NIGHT_ONLY, "Choose a subject with two nights"

## Download

`fetch_data` downloads the recordings from PhysioNet. For each night we get two files:

- the **PSG** file (`...-PSG.edf`): the signals
- the **hypnogram** file (`...-Hypnogram.edf`): the stages scored by the expert

With `path=DATA_DIR` the files are saved on your Drive. The second time you run this cell, nothing is downloaded. The first time it takes one or two minutes per night.

In [ ]:
# Download nights 1 and 2 of SUBJECT_A (or find them on Drive).
# The result is a list with one pair [PSG file, hypnogram file] for each night
files_A = fetch_data(subjects=[SUBJECT_A], recording=[1, 2], path=DATA_DIR)

# Print the names of the four files
for night, (psg_file, hyp_file) in enumerate(files_A, start=1):
    print(f"Night {night}")
    print("  signals:  ", os.path.basename(psg_file))
    print("  hypnogram:", os.path.basename(hyp_file))

**What we have now.** `files_A`: the paths of the four files on Drive. Nothing is in memory yet.

# Step 3 - Load and inspect one night

**Where we are.** Main path, stage 2 of 8: download ✓ -> **[read]** -> cut the wake -> epochs -> features -> split -> train -> test

We have files on Drive. At the end of this step night 1 is in memory, with the expert's stages attached. For now we work on night 1 only: night 2 goes through the same steps later, all at once (end of Step 5).

> **What we do:** read the signals of night 1, look at the channels and at a few seconds of signal.
>
> **Why:** look at the data first. Before any model we want to know which channels we have, at which sampling rate, for how long.

`read_raw_edf` reads an EDF file. Two arguments:

- `stim_channel="Event marker"`: this channel is not a signal, it is a marker
- `infer_types=True`: MNE reads the type of each channel from its name (`EEG Fpz-Cz` becomes an EEG channel called `Fpz-Cz`)

The file also contains a temperature channel, which we do not need: we drop it.

Then we read the hypnogram and attach it to the signals as **annotations**. An annotation is a piece of the recording with a start time (onset, in seconds), a duration, and a description: here, the stage scored by the expert, for example `Sleep stage 2`.

In [ ]:
# The two files of night 1
psg_1, hyp_1 = files_A[0]

# Read the signals. preload=True loads them in memory
raw_1 = mne.io.read_raw_edf(psg_1, stim_channel="Event marker", infer_types=True, preload=True)
# Drop the temperature channel: we do not use it
raw_1.drop_channels([ch for ch, kind in zip(raw_1.ch_names, raw_1.get_channel_types()) if kind == 'temperature'])

# Read the hypnogram: a list of annotations (onset, duration, stage)
annot_1 = mne.read_annotations(hyp_1)
# Attach it to the signals
raw_1.set_annotations(annot_1, emit_warning=False)

# A summary of the recording
raw_1.info

In [ ]:
# The channels, their types, the sampling rate and the duration
print("Channels and their types:")
for ch, ch_type in zip(raw_1.ch_names, raw_1.get_channel_types()):
    print(f"  {ch:20s} {ch_type}")
print()
print("Sampling rate:", raw_1.info['sfreq'], "Hz")
print(f"Duration: {raw_1.times[-1] / 3600:.1f} hours")

# The first annotations of the hypnogram
print()
print("First 5 annotations (onset in s, duration in s, stage):")
for onset, duration, desc in list(zip(annot_1.onset, annot_1.duration, annot_1.description))[:5]:
    print(f"  {onset:8.0f} {duration:8.0f}   {desc}")

**What to notice.** Two EEG channels, one EOG, one EMG, a respiration channel and the marker. The EEG is sampled at 100 Hz: 100 values per second. The recording lasts much longer than a night: the cassette recorder was started during the day. The hypnogram covers the whole recording with long annotations (the first one is many hours of W). We deal with this in Step 4.

## 30 seconds of signal

**Side look.** The model does not need this. We plot 30 seconds of signal to see what we are working with.

We plot the EEG, EOG and EMG channels, starting at the first epoch that the expert scored as stage 2 (so we are sure to look at sleep, not at the hours of wake). The signals are stored in volts: we multiply by 10^6 to get microvolts (µV), the usual unit for EEG.

In [ ]:
# Where to start: the onset (in seconds) of the first annotation "Sleep stage 2"
start = annot_1.onset[list(annot_1.description).index('Sleep stage 2')]
length = 30                          # 30 seconds, one epoch
sfreq = raw_1.info['sfreq']          # samples per second (100)

# Keep only the EEG, EOG and EMG channels, and take the 30 seconds (from sample start*sfreq)
signals = raw_1.copy().pick(['eeg', 'eog', 'emg'])
data, times = signals.get_data(start=int(start * sfreq), stop=int((start + length) * sfreq),
                               return_times=True)

# One row per channel, in microvolts
fig, axes = plt.subplots(len(signals.ch_names), 1, sharex=True, figsize=(12, 7))
for ax, ch, sig in zip(axes, signals.ch_names, data):
    ax.plot(times, sig * 1e6, lw=0.6, color='#2E6790')
    ax.set_ylabel(f"{ch}\n(µV)")
axes[-1].set_xlabel('time from the start of the recording (s)')
fig.suptitle('30 seconds of polysomnography (light sleep)')
plt.tight_layout()
plt.show()

**What to notice.** The EEG oscillates by some tens of µV. The EOG moves slowly: in light sleep the eyes roll slowly. The EMG in this dataset is not the raw muscle signal but its envelope (a smooth measure of muscle tone), which is why it looks so smooth.

**What we have now.** `raw_1`: the signals of night 1, with the hypnogram attached. `annot_1`: the hypnogram of night 1.

# Step 4 - The hypnogram, and cutting the wake

**Where we are.** Main path, stage 3 of 8: download ✓ -> read ✓ -> **[cut the wake]** -> epochs -> features -> split -> train -> test

We have the whole 22-hour recording. At the end of this step we keep only the night, plus 30 minutes before and after.

> **What we do:** draw the hypnogram of night 1, then keep only the part of the recording around the sleep.
>
> **Why:** the recording contains many hours of wake before and after the night. If we keep them, W becomes by far the largest class, and the model can look good just by recognising wake.

## Some definitions

First, a few names and two small functions we use for the rest of the lesson. This cell only defines things: it computes nothing.

- `STAGE_NAMES`: from the descriptions in the hypnogram file to our stage names (stages 3 and 4 become N3; M and ? are not listed, so they are discarded)
- `SCHEMES`: from the five stages to the classes of each `CLASS_SCHEME`
- `CLASS_ORDER`: the order of the classes in tables and plots
- `HYPNO_ORDER`: the order of the classes in a hypnogram, from bottom to top
- `hypnogram_from_annotations`: turns the annotations into one stage every 30 s, for plotting
- `plot_hypnogram`: draws a hypnogram

In [ ]:
# From the descriptions in the hypnogram file to our stage names
STAGE_NAMES = {
    'Sleep stage W': 'W',
    'Sleep stage 1': 'N1',
    'Sleep stage 2': 'N2',
    'Sleep stage 3': 'N3',
    'Sleep stage 4': 'N3',     # stages 3 and 4 are merged into N3
    'Sleep stage R': 'REM',
}

# From the five stages to the classes of each scheme
SCHEMES = {
    '5': {'W': 'W', 'N1': 'N1',    'N2': 'N2',    'N3': 'N3',   'REM': 'REM'},
    '4': {'W': 'W', 'N1': 'Light', 'N2': 'Light', 'N3': 'Deep', 'REM': 'REM'},
    '3': {'W': 'W', 'N1': 'NREM',  'N2': 'NREM',  'N3': 'NREM', 'REM': 'REM'},
}
# The order of the classes in tables and plots
CLASS_ORDER = {'5': ['W', 'N1', 'N2', 'N3', 'REM'],
               '4': ['W', 'Light', 'Deep', 'REM'],
               '3': ['W', 'NREM', 'REM']}
# The order of the classes in a hypnogram, from bottom to top
HYPNO_ORDER = {'5': ['N3', 'N2', 'N1', 'REM', 'W'],
               '4': ['Deep', 'Light', 'REM', 'W'],
               '3': ['NREM', 'REM', 'W']}


def hypnogram_from_annotations(annot):
    """One value every 30 s: the time (in hours) and the stage."""
    times, stages = [], []
    for onset, duration, desc in zip(annot.onset, annot.duration, annot.description):
        if desc in STAGE_NAMES:                       # skip M and ?
            for k in range(int(duration // 30)):      # an annotation of 90 s gives 3 values
                times.append(onset + 30 * k)
                stages.append(STAGE_NAMES[desc])
    return np.array(times) / 3600, np.array(stages)


def plot_hypnogram(ax, times_h, stages, order, **kwargs):
    """Draw a hypnogram: time on x, stage on y (W at the top)."""
    levels = [order.index(s) for s in stages]        # each stage becomes a height
    ax.step(times_h, levels, where='post', **kwargs)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order)
    ax.set_xlabel('time from the start of the recording (hours)')

## Cutting the wake

We keep the recording from **30 minutes before the first sleep epoch** to **30 minutes after the last sleep epoch**, as in the MNE tutorial. "Sleep" here means any stage other than W.

`crop_wake` finds the first and the last sleep annotation, and keeps only the annotations in between (plus the margins). Then we draw the hypnogram before and after the cut, to see what we removed.

In [ ]:
def crop_wake(annot, margin_min=30):
    """Keep only from margin_min minutes before the first sleep to margin_min minutes after the last."""
    # True for every annotation that is a sleep stage (not W, not M, not ?)
    is_sleep = [desc in STAGE_NAMES and desc != 'Sleep stage W' for desc in annot.description]
    sleep_idx = np.flatnonzero(is_sleep)
    first_sleep = annot.onset[sleep_idx[0]]                                    # start of the first sleep
    last_sleep = annot.onset[sleep_idx[-1]] + annot.duration[sleep_idx[-1]]   # end of the last sleep
    return annot.copy().crop(first_sleep - margin_min * 60, last_sleep + margin_min * 60)


annot_1_cropped = crop_wake(annot_1)

# The hypnogram before and after the cut, on the same time axis
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 6))
for ax, annot, title in [(axes[0], annot_1, 'Whole recording'),
                         (axes[1], annot_1_cropped, 'After cutting the wake')]:
    t, s = hypnogram_from_annotations(annot)
    plot_hypnogram(ax, t, s, HYPNO_ORDER['5'], color='#2E6790')
    ax.set_title(f'Hypnogram, subject {SUBJECT_A}, night 1: {title.lower()}')
plt.tight_layout()
plt.show()

**What to notice.** In the whole recording, long flat lines at W come before and after the night. After the cut, the night is in the middle and the wake is a small part. Look for the cycles: descents to N3 early in the night, REM episodes (the level just below W) becoming longer towards the morning.

**What we have now.** `annot_1_cropped`: the hypnogram of night 1, only around the night. The signals are not cut yet: that happens in the next step, when we take only the epochs inside these annotations.

# Step 5 - From a continuous signal to epochs

**Where we are.** Main path, stage 4 of 8: download ✓ -> read ✓ -> cut the wake ✓ -> **[epochs]** -> features -> split -> train -> test

We have a continuous signal and a cropped hypnogram. At the end of this step we have about a thousand windows of 30 s, each with its stage, for night 1 and also for night 2.

> **What we do:** cut the night into epochs of 30 seconds, each with its stage.
>
> **Why:** the epoch is the unit of the expert's scoring, so it is the unit of our table: one epoch = one row, one stage = one label.

Three steps:

1. `set_annotations` attaches the cropped hypnogram to the signals
2. `events_from_annotations(..., chunk_duration=30.)` creates one **event** every 30 s inside each annotation: an event is a time (in samples) and a number for the stage. An annotation of 90 s of N2 becomes three events
3. `mne.Epochs` cuts 30 s of signal starting at each event (`tmax = 30 - 1/sfreq`, so each epoch has exactly 3000 samples)

MNE wants the stages as numbers: `EVENT_ID` gives a number to each description in the file, `STAGE_ID` gives a name to each number.

We keep the EEG, EOG and EMG channels. Today we only use the EEG, but the others will be useful in Lesson 5.

In [ ]:
# The number of each stage in the hypnogram file (stages 3 and 4 get the same number)
EVENT_ID = {'Sleep stage W': 1, 'Sleep stage 1': 2, 'Sleep stage 2': 3,
            'Sleep stage 3': 4, 'Sleep stage 4': 4, 'Sleep stage R': 5}
# The name of each number
STAGE_ID = {'W': 1, 'N1': 2, 'N2': 3, 'N3': 4, 'REM': 5}

# 1. Attach the cropped hypnogram (the wake outside the night is now without annotation)
raw_1.set_annotations(annot_1_cropped, emit_warning=False)

# 2. One event every 30 s inside each annotation. M and ? are not in EVENT_ID, so they get no event
events_1, _ = mne.events_from_annotations(raw_1, event_id=EVENT_ID, chunk_duration=30.)

# 3. Cut 30 s of signal at each event
tmax = 30. - 1. / raw_1.info['sfreq']
epochs_1 = mne.Epochs(raw_1, events_1, event_id=STAGE_ID, tmin=0., tmax=tmax,
                      baseline=None,                      # no baseline correction
                      picks=['eeg', 'eog', 'emg'],         # keep these channels only
                      on_missing='warn',                  # do not fail if a stage is missing
                      preload=True)

epochs_1

**What to notice.** About a thousand epochs, each with 3000 samples per channel (30 s at 100 Hz), and the count of each stage. `on_missing='warn'` is there because some nights have no epoch of a stage (for example, older subjects can have almost no N3).

## The same for night 2, in one function

Night 2 needs exactly what we did for night 1: read (Step 3), cut the wake (Step 4), cut into epochs (Step 5). Instead of copying all those cells, we put the steps in one function, `load_night(subject, night)`, and call it for night 2. The plots are left out: the function only does the work.

A second small function, `epoch_stages`, gives the stage name of each epoch (MNE stores the stage as a number in the third column of `epochs.events`).

In [ ]:
# From the number back to the name of the stage
ID_TO_STAGE = {number: name for name, number in STAGE_ID.items()}


def load_night(subject, night):
    """Steps 3, 4 and 5 for one night: get the files, read, cut the wake, cut into 30-s epochs."""
    psg_file, hyp_file = fetch_data(subjects=[subject], recording=[night], path=DATA_DIR)[0]
    raw = mne.io.read_raw_edf(psg_file, stim_channel="Event marker", infer_types=True, preload=True)
    annot = crop_wake(mne.read_annotations(hyp_file))
    raw.set_annotations(annot, emit_warning=False)
    events, _ = mne.events_from_annotations(raw, event_id=EVENT_ID, chunk_duration=30.)
    tmax = 30. - 1. / raw.info['sfreq']
    return mne.Epochs(raw, events, event_id=STAGE_ID, tmin=0., tmax=tmax,
                      baseline=None, picks=['eeg', 'eog', 'emg'],
                      on_missing='warn', preload=True)


def epoch_stages(epochs):
    """The stage (W, N1, N2, N3, REM) of each epoch."""
    return np.array([ID_TO_STAGE[number] for number in epochs.events[:, 2]])


# Night 2, in one line
epochs_2 = load_night(SUBJECT_A, 2)
print("Night 1:", len(epochs_1), "epochs")
print("Night 2:", len(epochs_2), "epochs")

**What we have now.** `epochs_1` and `epochs_2`: the two nights cut into 30-s epochs, each with its stage. These are the examples for training and for testing. What is still missing is the table: we build it in Step 8. Steps 6 and 7 are side looks at these epochs.

# Step 6 - How frequent is each stage?

**Side look.** The model does not need this. We count the epochs of each stage. We come back to the main path in Step 8.

> **What we do:** count the epochs of each stage, in each night.
>
> **Why:** the classes are unbalanced. This decides which metrics we can trust (Step 11).

In [ ]:
# The share of each stage, for each night (value_counts with normalize=True gives proportions)
counts = pd.DataFrame({
    'night 1': pd.Series(epoch_stages(epochs_1)).value_counts(normalize=True),
    'night 2': pd.Series(epoch_stages(epochs_2)).value_counts(normalize=True),
}).reindex(CLASS_ORDER['5']).fillna(0)        # stages in the usual order; 0 if a stage is missing

print((counts * 100).round(1).astype(str) + ' %')

counts.plot.bar(rot=0, figsize=(8, 4), color=['#2E6790', '#D98030'])
plt.ylabel('share of the epochs')
plt.title(f'Sleep stages, subject {SUBJECT_A}')
plt.grid(axis='y', alpha=0.3)
plt.show()

**What to notice.** N1 is rare. On the whole Sleep-EDF, N2 is about 40% of the epochs and N1 about 7%; a single person can differ (a young subject, for example, has more N3). A model that always answered the most frequent stage would have an accuracy equal to the share of that stage, and would never find an N1. The two nights have similar, but not identical, proportions.

# Step 7 - What does each stage look like?

**Side look.** The model does not need this. We look at the signal of each stage, to choose the features of Step 8 with a reason.

> **What we do:** plot one epoch of each stage, and the average spectrum of each stage.
>
> **Why:** if the stages look different to us, a model has something to learn. The spectrum also tells us which features to build in Step 8.

## One epoch per stage

We take the epoch in the middle of each stage in night 1, channel Fpz-Cz. Same y axis for all, so the amplitudes can be compared.

In [ ]:
stages_1 = epoch_stages(epochs_1)
# The stages present in night 1 (a stage with no epoch cannot be plotted)
present = [s for s in CLASS_ORDER['5'] if (stages_1 == s).sum() > 0]

fig, axes = plt.subplots(len(present), 1, sharex=True, sharey=True, figsize=(12, 2 * len(present)))
t = np.arange(epochs_1.get_data().shape[-1]) / epochs_1.info['sfreq']     # 0 to 30 s
for ax, stage in zip(axes, present):
    # All the epochs of this stage, channel Fpz-Cz: shape (epochs, samples)
    stage_epochs = epochs_1[stage].get_data(picks='Fpz-Cz')[:, 0, :]
    middle = stage_epochs[len(stage_epochs) // 2]       # the one in the middle
    ax.plot(t, middle * 1e6, lw=0.6, color='#2E6790')
    ax.set_ylabel(f'{stage}\n(µV)')
axes[0].set_ylim(-150, 150)
axes[-1].set_xlabel('time (s)')
fig.suptitle('One epoch of each stage (Fpz-Cz)')
plt.tight_layout()
plt.show()

**What to notice.** N3 has large, slow waves. W and REM look alike: fast, small. In N2 you may see a short burst of fast oscillation (a spindle) or a single large wave (a K-complex). One epoch is only one example: the next plot averages all of them.

## The average spectrum of each stage

The **power spectral density (PSD)** says how much power the signal has at each frequency: it decomposes the 30 s of signal into slow and fast oscillations, and says how strong each one is. We compute it with Welch's method for every epoch, between 0.5 and 30 Hz, and average the epochs of each stage. The y axis is logarithmic, because power falls quickly with frequency. The dashed lines are the limits of the bands we use in Step 8.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for stage in present:
    # The spectrum of every epoch of this stage, channel Fpz-Cz
    spectrum = epochs_1[stage].compute_psd(method='welch', fmin=0.5, fmax=30., picks='Fpz-Cz')
    psds, freqs = spectrum.get_data(return_freqs=True)      # shape: (epochs, channels, frequencies)
    # The average over the epochs of this stage
    ax.plot(freqs, psds[:, 0, :].mean(axis=0), label=stage)

ax.set_yscale('log')
ax.set_xlabel('frequency (Hz)')
ax.set_ylabel('power (V²/Hz, log scale)')
ax.set_title('Average spectrum of each stage, Fpz-Cz, night 1')
for f in [4.5, 8.5, 11.5, 15.5]:
    ax.axvline(f, color='grey', lw=0.5, ls='--')     # the band limits of Step 8
ax.legend()
plt.show()

**What to notice.** N3 has the most power at low frequencies (delta). N2 has a small bump between 11.5 and 15.5 Hz: the spindles. W has more power at high frequencies. N1 and REM lie between the others and close to each other: we can expect the model to confuse them.

This is the idea for the features: the stages differ in **how the power is spread across the frequencies**. In Step 8 we summarise each epoch with a few numbers that describe exactly this.

# Step 8 - Features: relative power in each band

**Where we are.** Main path, stage 5 of 8: download ✓ -> read ✓ -> cut the wake ✓ -> epochs ✓ -> **[features]** -> split -> train -> test

We have epochs of 3000 samples per channel: too many numbers, and not comparable one by one. At the end of this step each epoch is one row of a table with 10 numbers and its stage.

> **What we do:** turn each epoch into a few numbers: the share of its power in each frequency band, for each EEG channel.
>
> **Why:** the model needs a table. Step 7 showed that the stages differ in how the power is spread across frequencies, so this is a natural description of an epoch.

The bands are the ones of the MNE tutorial:

| Band | Range (Hz) |
|---|---|
| delta | 0.5-4.5 |
| theta | 4.5-8.5 |
| alpha | 8.5-11.5 |
| sigma | 11.5-15.5 |
| beta | 15.5-30 |

For each epoch and each channel:

1. compute the spectrum (as in Step 7)
2. divide it by its total, so that it sums to 1
3. add up the values inside each band

We get the **relative power**: the share of the power in each band. The five numbers of a channel sum to 1. Relative power does not depend on the overall amplitude of the signal, which changes between people and between electrodes.

2 EEG channels x 5 bands = **10 features** per epoch. Only the EEG is used: the EOG and the EMG stay in the epochs, but not in the table.

`extract_features(epochs)` does this for all the epochs of a night, and adds the column `stage` (the label), already converted to the classes of `CLASS_SCHEME`.

In [ ]:
# The frequency bands, in Hz
BANDS = {'delta': (0.5, 4.5), 'theta': (4.5, 8.5), 'alpha': (8.5, 11.5),
         'sigma': (11.5, 15.5), 'beta': (15.5, 30.)}


def extract_features(epochs):
    """One row per epoch: relative power in each band, for each EEG channel, plus the stage."""
    # 1. The spectrum of every epoch, EEG channels only
    spectrum = epochs.compute_psd(method='welch', picks='eeg', fmin=0.5, fmax=30.)
    psds, freqs = spectrum.get_data(return_freqs=True)     # (epochs, channels, frequencies)
    # 2. Divide by the total: each spectrum now sums to 1
    psds = psds / psds.sum(axis=-1, keepdims=True)

    # 3. Add up the values inside each band, for each channel
    columns = {}
    for ch_idx, ch in enumerate(spectrum.ch_names):
        for band, (fmin, fmax) in BANDS.items():
            in_band = (freqs >= fmin) & (freqs < fmax)      # True for the frequencies of this band
            columns[f'{ch}_{band}'] = psds[:, ch_idx, in_band].sum(axis=-1)

    # The table, plus the label
    df = pd.DataFrame(columns)
    df['stage'] = [SCHEMES[CLASS_SCHEME][s] for s in epoch_stages(epochs)]
    return df


df_1 = extract_features(epochs_1)
df_2 = extract_features(epochs_2)

print("Night 1:", df_1.shape, "| Night 2:", df_2.shape)
df_1.head()

**What to notice.** Each row is one epoch, each column a feature, the last column is the label. The names say where each feature comes from: `Fpz-Cz_delta` is the share of the power of Fpz-Cz in the delta band. This is the same kind of table as the Titanic of Lesson 3: from here on, the analysis is standard machine learning.

**What we have now.** `df_1` and `df_2`: one table per night, 10 features and the stage.

**Side look.** The model does not need this. Before training, a quick check: does a single feature already separate the stages?

A boxplot of the relative delta power of Fpz-Cz for each stage:

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_1, x='stage', y='Fpz-Cz_delta', order=CLASS_ORDER[CLASS_SCHEME], color='#9BBBD4')
plt.ylabel('relative delta power (Fpz-Cz)')
plt.title('Relative delta power for each stage, night 1')
plt.grid(axis='y', alpha=0.3)
plt.show()

**What to notice.** N3 is the highest, as expected, and N2 comes next. W is a surprise: its relative delta is often high, and it varies a lot. Fpz-Cz is on the forehead, close to the eyes: in wake, eye movements and blinks produce slow, large deflections that end up in the delta band. A feature does not measure only what its name says. The boxes overlap a lot: one feature is not enough, and the model will combine all ten.

# Step 9 - Train and test

**Where we are.** Main path, stage 6 of 8: download ✓ -> read ✓ -> cut the wake ✓ -> epochs ✓ -> features ✓ -> **[split]** -> train -> test

We have two tables. At the end of this step we have the four objects of every supervised analysis: features and labels, for train and for test.

> **What we do:** night 1 is the train set, night 2 is the test set.
>
> **Why:** this is the right split, by what we saw in the lesson on data leakage. Consecutive epochs of the same night are very similar, so a random split would put near-copies in train and test. The test set must be a different recording.

As in Lesson 3: `X` is the table of features, `y` is the label.

In [ ]:
# Night 1 = train, night 2 = test. X: the 10 features. y: the stage
X_train = df_1.drop(columns='stage')
y_train = df_1['stage']
X_test = df_2.drop(columns='stage')
y_test = df_2['stage']

print("Train (night 1):", X_train.shape)
print("Test  (night 2):", X_test.shape)

# Step 10 - The model

**Where we are.** Main path, stage 7 of 8: download ✓ -> read ✓ -> cut the wake ✓ -> epochs ✓ -> features ✓ -> split ✓ -> **[train]** -> test

We have the train set. At the end of this step we have a fitted model.

> **What we do:** fit a random forest on night 1.
>
> **Why:** a random forest works well on small tables of features, needs little tuning, and tells us which features it uses.

A random forest does not need scaled features. We still put it in a `Pipeline`, as in Lesson 3: if we later add a preprocessing step (a scaler, a feature selection), it goes inside the pipeline and learns from the train set only.

`n_estimators=100`: 100 trees. `random_state=42`: the seed, so that everyone gets the same model. `n_jobs=-1`: use all the CPU cores.

In [ ]:
model = make_pipeline(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
# Fit on night 1 only
model.fit(X_train, y_train)
model

# Step 11 - Evaluation on night 2

**Where we are.** Main path, stage 8 of 8: download ✓ -> read ✓ -> cut the wake ✓ -> epochs ✓ -> features ✓ -> split ✓ -> train ✓ -> **[test]**

We have a fitted model. In this step we use night 2 for the first time, and we get the result of the lesson.

> **What we do:** predict the stages of night 2 and compare them with the expert's.
>
> **Why:** night 2 was never used for training. This is our estimate of how the model scores a new night of the same person.

Two numbers:

- **accuracy**: the share of epochs with the right stage. With unbalanced classes it is dominated by the frequent stages.
- **balanced accuracy**: the recall of each class, averaged. Every stage counts the same, so a missed N1 is visible.

Then the `classification_report`: precision, recall and F1 for each stage, as in Lesson 3.

In [ ]:
# The predicted stage of every epoch of night 2
y_pred = model.predict(X_test)

print(f"Accuracy:          {accuracy_score(y_test, y_pred):.3f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, labels=CLASS_ORDER[CLASS_SCHEME], digits=3, zero_division=0))

## The confusion matrix

Rows are the true stages, columns the predicted ones. The diagonal holds the correct answers, the rest of each row says where the errors of that stage go.

We draw it in two ways:

- **counts**: how many epochs fall in each cell. The rows of the frequent stages (N2) have large numbers, those of N1 small ones.
- **normalised by row**: each row is divided by its total, so it sums to 1. The diagonal is the recall of each stage, and rare and frequent stages can be compared.

And for two sets:

- **night 1 (train)**: the epochs the model was fitted on. This is not an estimate of anything: it only shows how much the model has memorised.
- **night 2 (test)**: the honest evaluation.

In [ ]:
# The predictions on the training night, only for the comparison
y_pred_train = model.predict(X_train)
labels = CLASS_ORDER[CLASS_SCHEME]

# Rows: train, test. Columns: counts, normalised by row
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for row, (y_true, y_hat, name) in enumerate([(y_train, y_pred_train, 'Night 1 (train)'),
                                             (y_test, y_pred, 'Night 2 (test)')]):
    # left: counts of epochs
    ConfusionMatrixDisplay.from_predictions(y_true, y_hat, labels=labels,
                                            cmap='Blues', colorbar=False, ax=axes[row, 0])
    axes[row, 0].set_title(f'{name}: counts')
    # right: each row normalised to sum to 1
    ConfusionMatrixDisplay.from_predictions(y_true, y_hat, labels=labels,
                                            normalize='true', values_format='.2f',
                                            cmap='Blues', colorbar=False, ax=axes[row, 1])
    axes[row, 1].set_title(f'{name}: each row sums to 1')
plt.tight_layout()
plt.show()

**What to notice.** On night 1 the matrix is almost perfectly diagonal: a random forest with fully grown trees memorises its training epochs. The gap between night 1 and night 2 is the overfitting of Lesson 3, and it is the reason why only night 2 counts.

On night 2, N1 is the weak stage: most of its epochs are predicted as a neighbouring stage (N2, REM, W), as the spectra of Step 7 suggested. This is also the stage on which human experts disagree most. Look also at N3: where do its errors go? Compare accuracy and balanced accuracy: most of the gap comes from N1.

**The main path ends here.** We have a model and its honest score on a new night. Steps 12 and 13 look inside the result.

# Step 12 - Predicted vs true hypnogram

**Side look.** The model does not need this. We draw the predictions of Step 11 as a hypnogram, to see where in the night the errors are.

> **What we do:** draw the hypnogram scored by the expert and the one predicted by the model, for night 2.
>
> **Why:** the metrics are averages. The hypnogram shows *where* in the night the errors are.

The time of each epoch comes from the epochs themselves, not from the feature table: the position in the night is **not** a feature.

In [ ]:
# The time of each epoch of night 2, in hours (events[:, 0] is the first sample of the epoch)
times_h = epochs_2.events[:, 0] / epochs_2.info['sfreq'] / 3600
order = HYPNO_ORDER[CLASS_SCHEME]

# Top: the expert. Bottom: the model
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(12, 6))
plot_hypnogram(axes[0], times_h, y_test.to_numpy(), order, color='#2E6790')
axes[0].set_title('Night 2: scored by the expert')
plot_hypnogram(axes[1], times_h, y_pred, order, color='#D98030')
axes[1].set_title('Night 2: predicted by the model')
plt.tight_layout()
plt.show()

**What to notice.** The overall structure of the night is there: the cycles, the deep sleep at the start, the REM episodes. The predicted hypnogram is more fragmented: the model scores each epoch alone, while an expert also looks at the epochs around it. Where are the errors concentrated: in stable periods, or at the transitions between stages?

# Step 13 - What does the model use?

**Side look.** The model does not need this. We look inside the fitted model, to check that it relies on features that make sense.

> **What we do:** look at the feature importance of the random forest.
>
> **Why:** to check that the model uses features that make physiological sense.

`feature_importances_` measures how much each feature was used in the splits of the trees. As in Lesson 3: importance describes the model, not the physiology. It says that a feature carries information about the stage, not that it causes it.

In [ ]:
# The random forest inside the pipeline
rf = model.named_steps['randomforestclassifier']
# One importance per feature, sorted
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values()

importances.plot.barh(figsize=(7, 5), color='#2E6790')
plt.xlabel('importance (mean decrease in impurity)')
plt.title('Random forest: feature importance')
plt.grid(axis='x', alpha=0.3)
plt.show()

**What to notice.** Do the important features agree with Step 7? Delta should matter for N3, sigma for N2 (spindles), alpha and beta for W. If a feature with no clear physiological role were at the top, we would want to understand why.

# Step 14 - Questions to reason about

## Question 1 - Why do we cut the wake?

In Step 4 we removed the hours of wake before and after the night. Suppose we had kept them. What would happen to the accuracy and the balanced accuracy? Would the model be better?

<details>
<summary><b>Answer</b></summary>
<br>

The accuracy would certainly go up. W would become by far the largest class, and most of the added epochs are easy (the subject is awake and active, far from sleep onset), so the model gets them right and they dominate the average.

The balanced accuracy weighs each stage the same, so the many extra W epochs count only through the recall of W, which would rise a little: the balanced accuracy would probably rise much less than the accuracy. By how much cannot be known without running it: it depends on how many hours of wake the recording contains and on how easy they are.

The model would not be better at scoring sleep. The training would also change: with many more W examples, the model may answer W more often on borderline epochs, such as N1 near sleep onset. So the scores on the sleep stages could even get worse while the overall number improves. This is a general lesson: a metric can improve because the task became easier, not because the model did.
</details>

## Question 2 - Accuracy and balanced accuracy

With 5 classes the accuracy is higher than the balanced accuracy. Why? Is this always the case?

<details>
<summary><b>Answer</b></summary>
<br>

The accuracy weighs each epoch the same, so it is dominated by the frequent stages, such as N2, which are recognised well. The balanced accuracy is the mean of the recalls of the stages, so the poor recall of N1, a rare stage with few epochs, counts as much as the good recall of N2.

It is not always the case. The accuracy is higher when the frequent classes are recognised better than the rare ones, which is what usually happens, since the model has more examples of them. If a model recognised the rare classes well and the frequent ones badly, the balanced accuracy would be the higher of the two. The gap between the two numbers is itself information: a large gap says that the performance differs a lot between frequent and rare stages.
</details>

## Question 3 - Which stage is hard, and why?

N1 is often predicted as N2, REM or W, but almost never as N3. Why? Are there other reasons why N1 is hard?

<details>
<summary><b>Answer</b></summary>
<br>

The main reason is in the spectra of Step 7. N1, W and REM all show low-amplitude activity with mixed frequencies, and their spectra are close. N1 and N2 differ mostly by short events (an N2 epoch needs a spindle or a K-complex), which a spectrum averaged over 30 s can blur. N3 is dominated by large slow waves (delta), so its spectrum is very different. The model confuses stages whose features are similar.

Other reasons add up:

- N1 is rare, so the model has few examples to learn it.
- N1 is a transition: an epoch of 30 s scored N1 often contains a bit of wake or of N2.
- The label itself is uncertain: N1 is the stage on which human experts disagree most. Part of what we count as model errors may be disagreement with one particular expert.
- In REM the eye movements are a key sign, and in W the muscle tone; we use only the EEG, so the model does not see the EOG and the EMG.

How much each reason contributes cannot be known from these results alone.
</details>

## Question 4 - Fewer classes

We switch to `CLASS_SCHEME = "4"` (N1 and N2 merged into Light). What do you expect for the accuracy and the balanced accuracy? Can we compare the new numbers with the old ones?

<details>
<summary><b>Answer</b></summary>
<br>

The confusions between N1 and N2 are no longer errors, and the hardest class, N1, no longer has its own recall. So the balanced accuracy will most likely go up. The confusions of N1 with W and REM remain, now as errors of Light, and they are diluted among the many N2 epochs.

The accuracy will also go up, since the epochs where N1 and N2 were swapped now count as correct. How large the two changes are cannot be predicted without running it: it depends on how many N1 / N2 confusions the model made.

The new numbers are not comparable with the old ones. The task is easier, so a higher score does not mean a better model. A fair comparison needs the same classes. This matters when reading papers: sleep staging results with 3, 4 or 5 classes are not on the same scale.
</details>

## Question 5 - Relative or absolute power?

Why did we use the share of power in each band, and not the absolute power? What do we lose?

<details>
<summary><b>Answer</b></summary>
<br>

The absolute power depends on the overall amplitude of the signal, which changes for reasons unrelated to sleep: the electrode impedance, its exact position, the thickness of the skull, and these change between people and also between nights of the same person. The relative power describes the shape of the spectrum, which is what distinguishes the stages, and is less sensitive to the overall amplitude.

What we lose:

- The amplitude itself is informative: N3 is defined by large waves. The share of delta keeps part of this information, but not all.
- The five relative powers of a channel sum to 1, so one of them is redundant: if we know four, we know the fifth.

Would absolute power work better? Between two nights of the same person, maybe: the amplitude is fairly stable within a person. Between different people, probably worse. Without trying, we cannot know.
</details>

## Question 6 - Why night 1 and night 2?

Why not mix the epochs of both nights and split them at random, 70% train and 30% test? And what does the night 1 / night 2 split not tell us?

<details>
<summary><b>Answer</b></summary>
<br>

Consecutive epochs of the same night are very similar. A random split would put near-copies of the test epochs in the train set, and the score would be too optimistic. This is the leakage of the lesson on data leakage. A different night is a truly new recording.

What it does not tell us: the test is still on the same person, with the same electrodes, the same anatomy, probably similar sleep habits. The score answers "how well does the model score a new night of this person?", not "how well does it score a new person?". It is also one single test night: we have no spread, and another night could give a different number.
</details>

## Question 7 - Train on night 2, test on night 1

If we swapped the two nights (train on night 2, test on night 1), would we get the same scores?

<details>
<summary><b>Answer</b></summary>
<br>

Almost certainly not the same, and the direction cannot be known in advance. The two nights differ in their proportions of stages, in the amount of N1 and of transitions, in the quality of the signal. A night with more N1 and more fragmented sleep is harder to score, whether it is in the train or in the test set.

The difference between the two results would be a rough idea of how much a score from one single test night can vary. If it were large, a single number from one night would not be very informative.
</details>

## Question 8 - A fragmented hypnogram

The predicted hypnogram jumps between stages more often than the true one. Why? How could we fix it, and what could go wrong?

<details>
<summary><b>Answer</b></summary>
<br>

The model scores each epoch alone. An expert also looks at the epochs before and after: a single N1 epoch in the middle of a long N2 period is unlikely, and the scoring rules use the context explicitly.

Two ways to use the context:

- add features computed on the neighbouring epochs (for example the delta power of the previous and the next epoch);
- smooth the predictions over time, for example with a majority vote over a few epochs.

What could go wrong: smoothing can remove real short events, such as brief awakenings, which the expert did score. Whether the result improves overall cannot be known without trying. And the features of the neighbours must come from the signal, never from the labels of the neighbours: on a new night we do not have the labels.
</details>

## Question 9 - The time of the night as a feature

N3 is frequent early in the night and REM late. Would it help to add the time of each epoch as a feature? Is it a good idea?

<details>
<summary><b>Answer</b></summary>
<br>

It would probably raise the scores on night 2, but it is risky, for three reasons.

- Our time axis depends on the cut of Step 4, which starts 30 minutes before the first sleep epoch. We found that epoch using the hypnogram, so the feature would carry information from the labels. On a new recording without a hypnogram, we could not compute it in the same way.
- It does not describe the brain activity of the epoch. It is a statistical regularity of how nights usually go.
- The timing of the night changes between people (age, habits) and between nights (a late bedtime, a night with many awakenings). A model leaning on it can fail exactly on the unusual nights, which are often the interesting ones in a clinic.

A version computed from the signal only (for example, the time from the start of the recording) avoids the first problem, not the other two.
</details>

## Question 10 - What does the importance say?

In the plot of Step 13, the beta and alpha bands are at the top, and the sigma bands, the band of the spindles, are near the bottom. Does this mean that spindles do not matter for N2? And that beta activity is what defines the stages? How much can we trust this ranking?

<details>
<summary><b>Answer</b></summary>
<br>

Neither conclusion follows. The importance says how much the model used a feature in its splits, not what defines a stage, and not what causes it.

Why beta and alpha can be at the top: they separate W (and partly REM and N1) from the other stages, so they are useful in many splits of many trees. A feature that helps with several stages collects importance from all of them.

Why sigma can be low even if spindles define N2: the spindles are short (about 1 s in an epoch of 30 s), so their share of the power over the whole epoch is small. The model can also recognise N2 in other ways: not enough delta for N3, not enough beta for W. If other features already carry the same information, sigma adds little, and gets little importance.

How much to trust the ranking:

- The importance of a random forest (mean decrease in impurity) is computed on the train set, and it is known to be biased in some cases.
- Correlated features share their importance: Fpz-Cz_sigma and Pz-Oz_sigma carry similar information, so each can look less important than it is.
- The relative powers of a channel sum to 1, so they are not independent.
- It comes from one night of one person: with another seed, another night or another person, the ranking could change. From these results alone we cannot say how stable it is.
</details>

## Question 11 - Can the model be better than the expert?

Our labels are the scoring of one expert. Could the model be better than the expert? Could we measure it?

<details>
<summary><b>Answer</b></summary>
<br>

With these data we cannot measure it. Every score compares the model with the expert's labels, so the expert is right by definition. Where the model disagrees with the expert, we count an error even if the model were right.

Since experts disagree with each other on part of the epochs, the labels contain some noise. This puts a ceiling on the scores we can observe: a perfect model of "the true stage" would still not reach a perfect score against one expert. To say that a model is better than an expert we would need several experts scoring the same nights, and a way to define the reference from them, for example their consensus.
</details>

## Question 12 - As good as an expert?

Suppose the model agrees with the expert on about 80% of the epochs of night 2, roughly as often as two human experts agree with each other. Can we say that the model scores sleep as well as an expert?

<details>
<summary><b>Answer</b></summary>
<br>

Not yet. The model was trained and tested on the same person, on one night each. The agreement between experts is measured on many people and many nights. One night also gives one number with no spread: we do not know how much it would change on another night.

To compare, we need to test the model on many people it has never seen. Whether it would keep the same agreement cannot be known from today's results. This is the topic of Lesson 5.
</details>

# Before the next lesson

Our model was trained on one night of one person and tested on another night of **the same person**.

Think about these questions for Lesson 5:

1. Would this model work on a different person?
2. Why might it not work?
3. What changes in the signal between two different people? Think about age, the position of the electrodes, the thickness of the skull, the habits of sleep.

## Sources

- Sleep-EDF Database Expanded, PhysioNet: https://physionet.org/content/sleep-edfx/
- MNE tutorial, Sleep stage classification from polysomnography (PSG) data: https://mne.tools/stable/auto_tutorials/clinical/60_sleep.html
- scikit-learn documentation: `balanced_accuracy_score`